In [2]:
import pandas as pd
import numpy as np
import re

In [57]:
def largest_shareholder(owner_str):
    if pd.isna(owner_str) or str(owner_str).strip().lower() == 'unknown':
        return 'unknown'
    
    matches = re.findall(r'(.+?)\s*\[(\d+(?:\.\d+)?)%\]', owner_str)
    if not matches:
        return owner_str  # return as-is if no percentage found
    
    return max(matches, key=lambda x: float(x[1]))[0].strip()

def parse_owners(owner_str):
    if pd.isna(owner_str) or str(owner_str).strip() == '-':
        return {}
    
    # Try to find name [XX%] pairs
    matches = re.findall(r'([^;]+?)\s*\[(\d+(?:\.\d+)?)%\]', owner_str)
    
    if matches:
        # Sort by percentage descending
        matches = sorted(matches, key=lambda x: float(x[1]), reverse=True)
        result = {}
        ranks = ['Primary', 'Secondary', 'Third', 'Fourth', 'Fifth']
        for i, (name, pct) in enumerate(matches):
            label = ranks[i] if i < len(ranks) else f'Owner_{i+1}'
            result[f'{label} Project Owner'] = name.strip()
            result[f'{label} Ownership (%)'] = float(pct)
        return result
    else:
        # No percentages — treat whole string as primary owner
        return {'Primary Project Owner': owner_str.strip(),
                'Primary Ownership (%)': 100.0 } # ← only one name, so assumed 100%
    
def parse_ownersID(owner_str):
    if pd.isna(owner_str) or str(owner_str).strip() == '-':
        return {}
    
    # Try to find name [XX%] pairs
    matches = re.findall(r'([^;]+?)\s*\[(\d+(?:\.\d+)?)%\]', owner_str)
    
    if matches:
        # Sort by percentage descending
        matches = sorted(matches, key=lambda x: float(x[1]), reverse=True)
        result = {}
        ranks = ['Primary', 'Secondary', 'Third', 'Fourth', 'Fifth']
        for i, (name, pct) in enumerate(matches):
            label = ranks[i] if i < len(ranks) else f'Owner_{i+1}'
            result[f'{label} Project Owner ID'] = name.strip()
        return result
    else:
        # No percentages — treat whole string as primary owner
        return {'Primary Project Owner ID': owner_str.strip()} # ← only one name, so assumed 100%
    
def utility_factor(dataset):
    prod_col = 'Production 2024 (ttpa)'
    col_2023 = 'Production 2023 (ttpa)'
    col_2022 = 'Production 2022 (ttpa)'
    cap_col  = 'Design capacity (ttpa)'

    def to_num(val):
        try:
            return float(val)
        except:
            return np.nan

    # Cap each year's production at capacity
    def cap_at_capacity(row, col):
        prod = to_num(row[col])
        cap  = to_num(row[cap_col])
        if pd.notna(prod) and pd.notna(cap) and prod/cap > 1.0 and prod/cap <= 1.3:
            return cap
        return row[col]

    for col in [prod_col, col_2023, col_2022]:
        dataset[col] = dataset.apply(lambda row, c=col: cap_at_capacity(row, c), axis=1)

    # Valid = not unknown, not NaN, not zero
    def is_valid(val):
        if pd.isna(val) or val == 'unknown':
            return False
        num = to_num(val)
        return pd.notna(num) and num != 0

    # Fill 2024 with first valid year (2024 → 2023 → 2022)
    def fill_production(row):
        for col in [prod_col, col_2023, col_2022]:
            if is_valid(row[col]):
                return row[col]
        return 'unknown'

    dataset[prod_col] = dataset.apply(fill_production, axis=1)

    prod_num    = dataset[prod_col].apply(to_num)
    cap_num     = dataset[cap_col].apply(to_num)
    valid       = prod_num.notna() & cap_num.notna()
    global_util = prod_num[valid].sum() / cap_num[valid].sum()

    def calc_utility(row):
        try:
            prod = float(row[prod_col])
            cap  = float(row[cap_col])
            if prod == 0 or prod / cap > 1.3:
                return global_util
            return prod / cap
        except:
            return global_util

    dataset['Utility factor_2024'] = dataset.apply(calc_utility, axis=1)
    return dataset



In [58]:
iron_mines = pd.read_excel('../data/GlobalTrackerData/iron/Global-Iron-Ore-Mines-Tracker-August-2025-V1.xlsx', sheet_name='Main Data')
iron_mines_owners = pd.read_excel('../data/GlobalTrackerData/iron/Global-Energy-Ownership-Tracker-February-2026-V1.xlsx', sheet_name='Iron Mine Ownership')

# Filter the dataframe to only include the Status column with the value of 'Operating', 'Proposed', 'mothballed'
iron_mines = iron_mines[iron_mines['Operating status'].isin(['operating', 'proposed', 'mothballed'])]

# Replace the status name 'Operating', 'Proposed', 'Mothballed' with 'operating', 'probale', 'highly_probable' respectively
iron_mines['Operating status'] = iron_mines['Operating status'].replace({
    'Operating': 'operating',
    'Proposed': 'proposed',
    'Mothballed': 'mothballed'
})

# Separate GPS into two columns, Latitude and Longitude
iron_mines[['Lat', 'Lon']] = iron_mines['Coordinates'].str.split(',',
    expand=True
).astype(float)

# Select the largest shareholder in the Owner column as the operator and add a column "Operator" to the dataframe
iron_mines['Operator'] = iron_mines['Owner'].apply(largest_shareholder)

# Parse the Owner column to extract up to 5 owners and their ownership percentages, and add these as new columns to the dataframe
owner_cols = iron_mines['Parent'].apply(lambda x: pd.Series(parse_owners(x)))
owner_cols_ID = iron_mines['Parent GEM Entity ID'].apply(lambda x: pd.Series(parse_ownersID(x)))
iron_mines = pd.concat([iron_mines, owner_cols, owner_cols_ID], axis=1)

# Loop over each xxx project owner ID column, for each column, loop over each row, find the country name in the iron_mines_owners dataframe using the Parent GEM Entity ID that matches the ID in each row, and add a new column with the country name for each owner column. 
# If the row is blank, leave it alone, if there is no match between the two dataframe, leave the cell value blank. There may be multiple matches in the iron_mines_owners dataframe for each owner ID, but we only keep the first matched country name for each owner ID.
# The drop column has the same name 'Parent GEM Entity ID' as the merge key, so we need to drop it after each merge to avoid duplicate columns in the dataframe
hq_lookup = (
    iron_mines_owners
    .drop_duplicates(subset='Parent GEM Entity ID')
    .set_index('Parent GEM Entity ID')['Parent Headquarters Country']
    .to_dict()
)

owner_id_cols = [col for col in iron_mines.columns if col.endswith('Project Owner ID')]
for col in list(iron_mines.columns):  # list() captures columns before loop modifies them
    if col.endswith('Project Owner ID'):
        rank = col.replace(' Project Owner ID', '')
        iron_mines[f'{rank} Owner Country'] = iron_mines[col].map(hq_lookup)


# Calculate utility factor
iron_mines = utility_factor(iron_mines)

# Drop columns
iron_mines = iron_mines.drop(columns=['Coordinate accuracy', 'Total resource (inferred, indicated and measured, thousand metric tonnes)', 
                                      'Start date', 'Stop date', 'Owner', 'Owner GEM Entity ID','Owner name in local language/script'])
# Reorder columns: put 'Lat' and 'Lon' after 'Coordinates', put 'Operator' after 'Operating status', put 'GEM wiki page URL' to the last column
# Update column name: change 'Operating status' to 'Status', 'Municipality' to 'City', 'Subnational unit' to 'State/Province', 'Country/Area' to 'Country', 'Design capacity (ttpa)' to 'Capacity_2025', 'Total reserves (proven and probable, thousand metric tonnes)' to 'Reserves(iron)'
iron_mines = iron_mines.rename(columns={
    'Operating status': 'Status',
    'Municipality': 'City',
    'Subnational unit': 'State/Province',
    'Country/Area': 'Country',
    'Design capacity (ttpa)': 'Capacity_2025',
    'Total reserves (proven and probable, thousand metric tonnes)': 'Reserves(iron)'
})
cols = iron_mines.columns.tolist()
cols.insert(cols.index('Coordinates') + 1, cols.pop(cols.index('Lat')))
cols.insert(cols.index('Coordinates') + 2, cols.pop(cols.index('Lon')))
cols.insert(cols.index('Status') + 1, cols.pop(cols.index('Operator')))
cols.append(cols.pop(cols.index('GEM wiki page URL')))
iron_mines = iron_mines[cols]


/Volumes/work/Projects/Archie_steel/.venv/lib/python3.10/site-packages/openpyxl/worksheet/_reader.py:329: UserWarning: Unknown extension is not supported and will be removed
  warn(msg)


In [ ]:
# Outout the dataframe to a new Excel file
# iron_mines.to_excel('../data/Processed_data/iron_mines_data.xlsx', index=False)